
# PHASE 3F — Exact Small-Instance Optimality-Gap Study

This notebook addresses Reviewer 3's request for **optimality bounds / optimality gaps where computationally feasible**.

## Why an exact dynamic-programming solver is used

For the submitted SDS-FDR decoder:

- every operated flight departs either at reopening time or after the model-defined separation from the previous operated flight;
- the separation values are 1.5, 2.0, or 2.5 minutes;
- cancelled flights do not become predecessors;
- cancellation penalties are additive;
- all scheduled departures in the submitted evening bank occur before reopening.

Therefore, for a small flight subset, the exact scalar optimum can be obtained by dynamic programming over:

1. the set of operated flights,
2. the last operated flight,
3. the current departure time in 0.5-minute units.

Flights omitted from the operated path are cancelled and receive exactly the same cancellation penalties as in the submitted decoder.

A dominance rule removes a state only when another state with the same operated set and same last flight has both:
- an earlier/equal departure time, and
- a lower/equal accumulated scalar cost.

Such a dominated state can never produce a better continuation because all operated cost terms are non-decreasing with delay.

## Exact-test instances

We use deterministic nested subsets of the authoritative 48-flight semi-synthetic CGK bank:

- N = 8
- N = 10
- N = 12
- N = 14
- N = 16

For each N, the subset contains the **earliest N scheduled flights** from the authoritative seed-7 bank. This rule is fixed in advance and does not cherry-pick favorable instances.

The normalized reference vector is recomputed for each subset using the same 300-random-schedule protocol as the main study.

## Heuristic comparison

For each exact instance:

- SA
- VNS
- AMQSA

are run for:

- 20 seeds (`2000–2019`);
- exactly 5,000 objective evaluations/run.

The main quantities reported are:

- exact scalar optimum;
- heuristic mean scalar gap;
- heuristic best scalar gap;
- worst scalar gap;
- exact-optimum hit rate;
- runtime.

The optimality gap is defined only for the **normalized scalar objective**, because that is the criterion solved exactly.


In [ ]:

# STEP L1 — Fresh Colab runtime
%pip install -q --force-reinstall "numpy==2.0.2"
print("NumPy 2.0.2 installation requested.")
print("RESTART RUNTIME now, then continue with STEP L2.")


In [ ]:

# STEP L2 — Environment lock
import sys, os, json, platform, time, math, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import matplotlib
import matplotlib.pyplot as plt

OUTPUT_DIR = Path("/content/phase3f_exact_optimality")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ENV = {
    "python": sys.version.replace("\n"," "),
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scipy": scipy.__version__,
    "matplotlib": matplotlib.__version__,
    "cpu_count": os.cpu_count(),
}
print(json.dumps(ENV, indent=2))

if np.__version__ != "2.0.2":
    raise RuntimeError("STOP: Phase 3F is locked to NumPy 2.0.2.")

with open(OUTPUT_DIR/"phase3f_environment.json","w") as f:
    json.dump(ENV,f,indent=2)

print("Environment check PASSED.")


In [ ]:

# STEP B3 — Definitions: SDS-FDR + fair-budget solvers
# Cell ini hanya mendefinisikan model/solver. Belum menjalankan 20-run experiment.

import time
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon, friedmanchisquare, rankdata

# -------------------------- RUN FLAGS --------------------------
RUN_CGK       = True
RUN_ABLATION  = True
RUN_BENCHMARK = True

# Optional visual analyses from the old notebook are intentionally omitted here.
# Phase 2 focuses on budget fairness + authoritative numerical outputs.

# -------------------------- FAIR BUDGET ------------------------
POP = 50
MAX_EVALS = 5_000
N_RUNS = 20

BD = 30
BPOP = 30
BENCH_MAX_EVALS = 10_000
BN_RUNS = 20

P_CANCEL_INIT = 0.05
QUENCH_FRAC = 0.60

CGK_GRID = np.arange(50, MAX_EVALS + 1, 50)       # 100 checkpoints
BENCH_GRID = np.linspace(100, BENCH_MAX_EVALS, 100).astype(int)

# -------------------------- INSTANCE ---------------------------
# Freeze the same semi-synthetic CGK bank used in the submitted experiment.
# A real/OpenSky disruption instance will be treated as a separate validation experiment.
USE_OPENSKY_FILE = False

RNG_DATA = np.random.default_rng(7)

REOPEN = 180.0
RECOVERY_CUTOFF = 260.0   # 22:20 experimental recovery cutoff/horizon
CP = 6_000.0
COMP_CANCEL = 300_000.0
REV_LOSS_FRAC = 0.7
PAX_CANCEL_LOSS = 1_300_000.0
CANCEL_EQUIV_MIN = 600.0

FLEET = {
    "B738":  (189, "M", "jet",  25_000, 60e6,  "dom"),
    "B739":  (215, "M", "jet",  27_000, 60e6,  "dom"),
    "A320":  (180, "M", "jet",  24_000, 60e6,  "dom"),
    "A333":  (290, "H", "jet",  65_000, 200e6, "intl"),
    "B77W":  (393, "H", "jet",  75_000, 250e6, "intl"),
    "ATR72": (72,  "M", "prop", 10_000, 30e6,  "reg"),
}
FARE = {"dom": (1.0e6, 1.5e6), "intl": (2.6e6, 3.8e6), "reg": (0.7e6, 1.1e6)}

OPERATORS = [
    ("GIA", "Garuda",        {"B738": 0.5, "A333": 0.3, "B77W": 0.2}),
    ("LNI", "Lion Air",      {"B738": 0.55, "B739": 0.45}),
    ("CTV", "Citilink",      {"A320": 1.0}),
    ("BTK", "Batik Air",     {"A320": 0.6, "B738": 0.4}),
    ("SJV", "Super Air Jet", {"A320": 1.0}),
    ("WON", "Wings Air",     {"ATR72": 1.0}),
    ("AWQ", "AirAsia Indonesia", {"A320": 1.0}),
    ("INTL", "Foreign carrier",  {"B77W": 0.6, "A333": 0.4}),
]
OP_WEIGHTS = [0.16, 0.28, 0.14, 0.12, 0.10, 0.06, 0.06, 0.08]
PREFIX2OP = {p: (name, fleet) for p, name, fleet in OPERATORS}
SIDS = ["CKG1A", "DOLTA1", "PURWO1", "NOKTA1"]


def _flight_row(i, prefix, airline, ac, sched):
    cap, wake, eng, cfm, repos, band = FLEET[ac]
    lo, hi = FARE[band]
    return {
        "flight": f"{prefix}{RNG_DATA.integers(100, 999)}",
        "airline": airline,
        "aircraft": ac,
        "capacity": cap,
        "wake": wake,
        "engine": eng,
        "sid": SIDS[int(RNG_DATA.integers(0, len(SIDS)))],
        "sched_min_from_1800": sched,
        "cfm_idr_per_min": cfm,
        "reposition_idr": repos,
        "fare_idr": float(np.round(RNG_DATA.uniform(lo, hi), -3)),
    }


def build_frozen_dataset():
    n = 48
    sched = np.sort(RNG_DATA.choice(np.arange(1, 180), size=n, replace=False))
    rows = []
    for i in range(n):
        k = RNG_DATA.choice(len(OPERATORS), p=OP_WEIGHTS)
        prefix, airline, fleet = OPERATORS[k]
        ac = RNG_DATA.choice(list(fleet), p=list(fleet.values()))
        rows.append(_flight_row(i, prefix, airline, ac, float(sched[i])))
    return pd.DataFrame(rows), "seeded semi-synthetic CGK evening bank (seed=7)"


FLIGHTS, DATA_SOURCE = build_frozen_dataset()
FLIGHTS.to_csv(OUTPUT_DIR / "phase2_cgk_instance.csv", index=False)

N = len(FLIGHTS)
SCHED = FLIGHTS["sched_min_from_1800"].to_numpy(float)
CAP = FLIGHTS["capacity"].to_numpy(float)
NP_SEATS = 0.75 * CAP
CFM = FLIGHTS["cfm_idr_per_min"].to_numpy(float)
FARE_V = FLIGHTS["fare_idr"].to_numpy(float)
REPOS = FLIGHTS["reposition_idr"].to_numpy(float)
WAKE = FLIGHTS["wake"].to_numpy()
ENG = FLIGHTS["engine"].to_numpy()
SID = FLIGHTS["sid"].to_numpy()


def _sep_rule(prev, nxt):
    s = 120.0 if SID[prev] == SID[nxt] else 90.0
    if WAKE[prev] == "H" and WAKE[nxt] == "M":
        s = max(s, 120.0)
    if ENG[prev] == "prop" and ENG[nxt] == "jet" and SID[prev] == SID[nxt]:
        s = max(s, 150.0)
    return s / 60.0


SEP_M = np.array([[_sep_rule(u, v) for v in range(N)] for u in range(N)])


def separation(prev, nxt):
    return SEP_M[prev, nxt]


def mu_disappointment(t):
    return np.clip(np.cbrt((t / 29.0) ** 2 / 29.0), 0.0, 1.0)


def decode(perm, cancel):
    f1 = f2 = f3 = 0.0
    delays = np.full(N, -1.0)
    prev = -1
    tcur = REOPEN
    ncan = 0

    for fl in perm:
        if cancel[fl]:
            f1 += (COMP_CANCEL + REV_LOSS_FRAC * FARE_V[fl]) * NP_SEATS[fl] + REPOS[fl]
            f2 += PAX_CANCEL_LOSS * NP_SEATS[fl]
            f3 += CANCEL_EQUIV_MIN
            ncan += 1
            continue

        dep = tcur if prev < 0 else tcur + separation(prev, fl)

        if dep > RECOVERY_CUTOFF:
            f1 += (COMP_CANCEL + REV_LOSS_FRAC * FARE_V[fl]) * NP_SEATS[fl] + REPOS[fl]
            f2 += PAX_CANCEL_LOSS * NP_SEATS[fl]
            f3 += CANCEL_EQUIV_MIN
            ncan += 1
            continue

        t = dep - SCHED[fl]
        delays[fl] = t
        f1 += t * CFM[fl] + mu_disappointment(t) * NP_SEATS[fl] * FARE_V[fl]
        f2 += NP_SEATS[fl] * CP * t
        f3 += t
        prev, tcur = fl, dep

    return f1, f2, f3, ncan, delays


# Freeze the same random-schedule normalization protocol.
_r = np.random.default_rng(0)
_samples = []
for _ in range(300):
    p = _r.permutation(N)
    c = _r.random(N) < 0.05
    _samples.append(decode(p, c)[:3])

F_REF = np.array(_samples).mean(axis=0)
W = np.array([1/3, 1/3, 1/3])


def fitness(perm, cancel):
    f1, f2, f3, ncan, _ = decode(perm, cancel)
    f = np.array([f1, f2, f3])
    return float(np.sum(W * f / F_REF)), f, ncan


# Priority operator from the submitted implementation.
WGT_PRI = NP_SEATS * CP + CFM + NP_SEATS * FARE_V / 300.0


# --------------------- BUDGET / CURVE RECORDER ---------------------
class CGKBudget:
    def __init__(self, max_evals=MAX_EVALS, grid=CGK_GRID):
        self.max_evals = int(max_evals)
        self.evals = 0
        self.grid = np.asarray(grid, int)
        self.points = []
        self.gi = 0

    @property
    def remaining(self):
        return self.max_evals - self.evals

    def evaluate(self, perm, cancel):
        if self.evals >= self.max_evals:
            raise RuntimeError("Evaluation budget exhausted")
        out = fitness(perm, cancel)
        self.evals += 1
        return out

    def observe(self, best):
        best_f1 = float(best[2][1][0])
        while self.gi < len(self.grid) and self.evals >= self.grid[self.gi]:
            self.points.append(best_f1)
            self.gi += 1

    def curve(self, best):
        self.observe(best)
        while len(self.points) < len(self.grid):
            self.points.append(float(best[2][1][0]))
        return np.asarray(self.points, float)


def _init_perm_pop(rng):
    return ([rng.permutation(N) for _ in range(POP)],
            [rng.random(N) < P_CANCEL_INIT for _ in range(POP)])


def _ox(rng, p1, p2):
    a, b = sorted(rng.choice(N, 2, replace=False))
    child = -np.ones(N, int)
    child[a:b] = p1[a:b]
    seg = set(p1[a:b])
    fill = [g for g in p2 if g not in seg]
    child[child < 0] = fill
    return child


def _update_best(best, p, c, fit):
    if best is None or fit[0] < best[2][0]:
        return (p.copy(), c.copy(), fit)
    return best


# -------------------------- FAIR CGK SOLVERS --------------------------
def run_ga(seed, max_evals=MAX_EVALS):
    rng = np.random.default_rng(seed)
    pc, pm = 1.0, 0.2
    budget = CGKBudget(max_evals)
    perms, cans = _init_perm_pop(rng)

    fits, best = [], None
    for p, c in zip(perms, cans):
        f = budget.evaluate(p, c)
        fits.append(f)
        best = _update_best(best, p, c, f)
        budget.observe(best)

    while budget.remaining >= POP:
        np_perms, np_cans, np_fits = [], [], []
        while len(np_perms) < POP:
            i1 = min(rng.choice(POP, 3, replace=False), key=lambda i: fits[i][0])
            i2 = min(rng.choice(POP, 3, replace=False), key=lambda i: fits[i][0])

            if rng.random() < pc:
                child = _ox(rng, perms[i1], perms[i2])
                mask = rng.random(N) < 0.5
                cbit = np.where(mask, cans[i1], cans[i2])
            else:
                child, cbit = perms[i1].copy(), cans[i1].copy()

            if rng.random() < pm:
                a, b = rng.choice(N, 2, replace=False)
                child[a], child[b] = child[b], child[a]

            if rng.random() < 0.1:
                k = rng.integers(0, N)
                cbit = cbit.copy()
                cbit[k] = ~cbit[k]

            f = budget.evaluate(child, cbit)
            np_perms.append(child)
            np_cans.append(cbit)
            np_fits.append(f)
            best = _update_best(best, child, cbit, f)
            budget.observe(best)

        perms, cans, fits = np_perms, np_cans, np_fits
        wi = int(np.argmax([f[0] for f in fits]))
        perms[wi], cans[wi], fits[wi] = best[0].copy(), best[1].copy(), best[2]

    assert budget.evals == max_evals
    return best, budget.curve(best), budget.evals


def run_sa(seed, max_evals=MAX_EVALS):
    rng = np.random.default_rng(seed)
    budget = CGKBudget(max_evals)

    cur_p = rng.permutation(N)
    cur_c = rng.random(N) < P_CANCEL_INIT
    cur_f = budget.evaluate(cur_p, cur_c)
    best = (cur_p.copy(), cur_c.copy(), cur_f)
    budget.observe(best)

    T, alpha = 0.02, 0.999

    while budget.remaining > 0:
        cp, cc = cur_p.copy(), cur_c.copy()
        u = rng.random()

        if u < 0.45:
            a, b = rng.choice(N, 2, replace=False)
            cp[a], cp[b] = cp[b], cp[a]
        elif u < 0.90:
            a, b = rng.choice(N, 2, replace=False)
            g = cp[a]
            cp = np.delete(cp, a)
            cp = np.insert(cp, b, g)
        else:
            k = rng.integers(0, N)
            cc[k] = ~cc[k]

        cf = budget.evaluate(cp, cc)
        d = cf[0] - cur_f[0]

        if d < 0 or rng.random() < np.exp(-d / max(T, 1e-12)):
            cur_p, cur_c, cur_f = cp, cc, cf
            best = _update_best(best, cp, cc, cf)

        T *= alpha
        budget.observe(best)

    assert budget.evals == max_evals
    return best, budget.curve(best), budget.evals


def _keys_decode(keys):
    return np.argsort(keys[:N], kind="stable"), keys[N:] > 0.9


def run_pso(seed, max_evals=MAX_EVALS):
    rng = np.random.default_rng(seed)
    D = 2 * N
    budget = CGKBudget(max_evals)

    x = rng.random((POP, D))
    v = np.zeros_like(x)
    pb = x.copy()

    pf, best = [], None
    for xi in x:
        p, c = _keys_decode(xi)
        f = budget.evaluate(p, c)
        pf.append(f)
        best = _update_best(best, p, c, f)
        budget.observe(best)

    gi = int(np.argmin([f[0] for f in pf]))
    gb, gf = pb[gi].copy(), pf[gi]

    total_iters = (max_evals - POP) // POP
    it = 0

    while budget.remaining >= POP:
        frac = it / max(total_iters - 1, 1)
        w = 0.9 - 0.5 * frac
        r1, r2 = rng.random((POP, D)), rng.random((POP, D))
        v = np.clip(w * v + 1.8 * r1 * (pb - x) + 1.8 * r2 * (gb - x), -0.5, 0.5)
        x = np.clip(x + v, 0, 1)

        for i in range(POP):
            p, c = _keys_decode(x[i])
            fi = budget.evaluate(p, c)

            if fi[0] < pf[i][0]:
                pb[i], pf[i] = x[i].copy(), fi
                if fi[0] < gf[0]:
                    gb, gf = x[i].copy(), fi

            best = _update_best(best, p, c, fi)
            budget.observe(best)

        it += 1

    assert budget.evals == max_evals
    p, c = _keys_decode(gb)
    solver_best = (p, c, gf)
    # 'best' and solver_best should agree on scalar objective.
    if best[2][0] < solver_best[2][0]:
        solver_best = best
    return solver_best, budget.curve(solver_best), budget.evals


def run_zoa(seed, max_evals=MAX_EVALS):
    rng = np.random.default_rng(seed)
    D = 2 * N
    budget = CGKBudget(max_evals)

    x = rng.random((POP, D))
    fits, best = [], None
    for xi in x:
        p, c = _keys_decode(xi)
        f = budget.evaluate(p, c)
        fits.append(f)
        best = _update_best(best, p, c, f)
        budget.observe(best)

    gi = int(np.argmin([f[0] for f in fits]))
    pz, pzf = x[gi].copy(), fits[gi]

    while budget.remaining > 0:
        for i in range(POP):
            if budget.remaining <= 0:
                break

            # Phase 1: foraging
            r = rng.random(D)
            I = 1 + rng.integers(0, 2)
            xn = np.clip(x[i] + r * (pz - I * x[i]), 0, 1)
            p, c = _keys_decode(xn)
            fn = budget.evaluate(p, c)

            if fn[0] < fits[i][0]:
                x[i], fits[i] = xn, fn
            if fits[i][0] < pzf[0]:
                pz, pzf = x[i].copy(), fits[i]

            bp, bc = _keys_decode(pz)
            best = _update_best(best, bp, bc, pzf)
            budget.observe(best)

            if budget.remaining <= 0:
                break

            # Phase 2: defense; schedule based on evaluation progress.
            progress = budget.evals / max_evals
            if rng.random() < 0.5:
                xn = np.clip(
                    x[i] + 0.01 * (2 * rng.random(D) - 1) * (1 - progress) * x[i],
                    0, 1
                )
            else:
                k = rng.integers(0, POP)
                r = rng.random(D)
                I = 1 + rng.integers(0, 2)
                xn = np.clip(x[i] + r * (x[k] - I * x[i]), 0, 1)

            p, c = _keys_decode(xn)
            fn = budget.evaluate(p, c)

            if fn[0] < fits[i][0]:
                x[i], fits[i] = xn, fn
            if fits[i][0] < pzf[0]:
                pz, pzf = x[i].copy(), fits[i]

            bp, bc = _keys_decode(pz)
            best = _update_best(best, bp, bc, pzf)
            budget.observe(best)

    assert budget.evals == max_evals
    p, c = _keys_decode(pz)
    solver_best = (p, c, pzf)
    if best[2][0] < solver_best[2][0]:
        solver_best = best
    return solver_best, budget.curve(solver_best), budget.evals


def _order_impose(rng, perm, guide, frac):
    s = max(2, int(np.ceil(max(frac, 1e-9) * N)))
    s = min(s, N)
    subset = rng.choice(N, s, replace=False)
    mask = np.isin(perm, subset)
    pos_g = np.empty(N, int)
    pos_g[guide] = np.arange(N)
    ordered = sorted(perm[mask], key=lambda f: pos_g[f])
    out = perm.copy()
    out[mask] = ordered
    return out


def run_dzoa(seed, max_evals=MAX_EVALS):
    rng = np.random.default_rng(seed)
    budget = CGKBudget(max_evals)
    perms, cans = _init_perm_pop(rng)

    fits, best = [], None
    for p, c in zip(perms, cans):
        f = budget.evaluate(p, c)
        fits.append(f)
        best = _update_best(best, p, c, f)
        budget.observe(best)

    gi = int(np.argmin([f[0] for f in fits]))
    pz_p, pz_c, pz_f = perms[gi].copy(), cans[gi].copy(), fits[gi]

    while budget.remaining > 0:
        for i in range(POP):
            if budget.remaining <= 0:
                break

            # Phase 1
            r = rng.random()
            I = 1 + rng.integers(0, 2)
            xp = _order_impose(rng, perms[i], pz_p, frac=0.25 * r * I)
            xc = cans[i].copy()
            m = rng.random(N) < 0.5 * r
            xc = np.where(m, pz_c, xc)

            fn = budget.evaluate(xp, xc)
            if fn[0] < fits[i][0]:
                perms[i], cans[i], fits[i] = xp, xc, fn
            if fits[i][0] < pz_f[0]:
                pz_p, pz_c, pz_f = perms[i].copy(), cans[i].copy(), fits[i]
            best = _update_best(best, pz_p, pz_c, pz_f)
            budget.observe(best)

            if budget.remaining <= 0:
                break

            # Phase 2
            progress = budget.evals / max_evals
            if rng.random() < 0.5:
                kswap = max(1, int(round(4 * (1 - progress))))
                xp = perms[i].copy()
                for _ in range(kswap):
                    a, b = rng.choice(N, 2, replace=False)
                    xp[a], xp[b] = xp[b], xp[a]
                xc = cans[i].copy()
                if rng.random() < 0.2:
                    j = rng.integers(0, N)
                    xc[j] = ~xc[j]
            else:
                k = rng.integers(0, POP)
                r = rng.random()
                I = 1 + rng.integers(0, 2)
                xp = _order_impose(rng, perms[i], perms[k], frac=0.25 * r * I)
                xc = cans[i].copy()
                m = rng.random(N) < 0.5 * r
                xc = np.where(m, cans[k], xc)

            fn = budget.evaluate(xp, xc)
            if fn[0] < fits[i][0]:
                perms[i], cans[i], fits[i] = xp, xc, fn
            if fits[i][0] < pz_f[0]:
                pz_p, pz_c, pz_f = perms[i].copy(), cans[i].copy(), fits[i]
            best = _update_best(best, pz_p, pz_c, pz_f)
            budget.observe(best)

    assert budget.evals == max_evals
    solver_best = (pz_p, pz_c, pz_f)
    if best[2][0] < solver_best[2][0]:
        solver_best = best
    return solver_best, budget.curve(solver_best), budget.evals


def run_amqsa(seed, max_evals=MAX_EVALS, T0=0.02, alpha=0.999,
              lam=0.9, quench_frac=QUENCH_FRAC):
    rng = np.random.default_rng(seed)
    budget = CGKBudget(max_evals)

    cur_p = rng.permutation(N)
    cur_c = rng.random(N) < P_CANCEL_INIT
    cur_f = budget.evaluate(cur_p, cur_c)
    best = (cur_p.copy(), cur_c.copy(), cur_f)
    budget.observe(best)

    T = T0
    wts = np.array([1.5, 1.5, 1.0, 1.0, 0.3])
    quenched = False

    while budget.remaining > 0:
        if not quenched and budget.evals >= quench_frac * max_evals:
            T = 1e-12
            quenched = True
            cur_p, cur_c, cur_f = best[0].copy(), best[1].copy(), best[2]

        pr = np.maximum(wts, 0.05)
        pr = pr / pr.sum()
        oi = rng.choice(5, p=pr)

        cp, cc = cur_p.copy(), cur_c.copy()

        if oi == 0:
            a, b = rng.choice(N, 2, replace=False)
            cp[a], cp[b] = cp[b], cp[a]
        elif oi == 1:
            a, b = rng.choice(N, 2, replace=False)
            g = cp[a]
            cp = np.delete(cp, a)
            cp = np.insert(cp, b, g)
        elif oi == 2:
            L = int(rng.integers(2, 4))
            a = int(rng.integers(0, N - L))
            blk = cp[a:a+L].copy()
            rest = np.concatenate([cp[:a], cp[a+L:]])
            b = int(rng.integers(0, len(rest) + 1))
            cp = np.concatenate([rest[:b], blk, rest[b:]])
        elif oi == 3:
            i, j = sorted(rng.choice(N, 2, replace=False))
            if WGT_PRI[cp[j]] <= WGT_PRI[cp[i]]:
                i, j = j, i
            cp[i], cp[j] = cp[j], cp[i]
        else:
            k = rng.integers(0, N)
            cc[k] = ~cc[k]

        cf = budget.evaluate(cp, cc)
        d = cf[0] - cur_f[0]
        acc = d < 0 or rng.random() < np.exp(-d / max(T, 1e-12))

        wts[oi] = lam * wts[oi] + (1-lam) * (1.0 if d < 0 else (0.3 if acc else 0.0))

        if acc:
            cur_p, cur_c, cur_f = cp, cc, cf
            best = _update_best(best, cp, cc, cf)

        T *= alpha
        budget.observe(best)

    assert budget.evals == max_evals
    return best, budget.curve(best), budget.evals


ALGOS = {
    "GA": run_ga,
    "SA": run_sa,
    "PSO": run_pso,
    "ZOA": run_zoa,
    "dZOA": run_dzoa,
    "AMQSA": run_amqsa,
}


# -------------------------- ABLATION FAIR BUDGET --------------------------
def _ablation_variant(seed, ops, w0, max_evals=MAX_EVALS,
                      T0=0.02, alpha=0.999, lam=0.9,
                      quench=QUENCH_FRAC, reheat=False,
                      stag_limit=400, rf=25.0):
    rng = np.random.default_rng(seed)
    budget = CGKBudget(max_evals)

    cur_p = rng.permutation(N)
    cur_c = rng.random(N) < P_CANCEL_INIT
    cur_f = budget.evaluate(cur_p, cur_c)
    best = (cur_p.copy(), cur_c.copy(), cur_f)

    T = T0
    wts = np.array(w0, float)
    quenched = False
    stag = 0

    while budget.remaining > 0:
        if not reheat and not quenched and budget.evals >= quench * max_evals:
            T = 1e-12
            quenched = True
            cur_p, cur_c, cur_f = best[0].copy(), best[1].copy(), best[2]

        pr = np.maximum(wts, 0.05)
        pr = pr / pr.sum()
        oi = rng.choice(len(ops), p=pr)
        op = ops[oi]

        cp, cc = cur_p.copy(), cur_c.copy()

        if op == "swap":
            a, b = rng.choice(N, 2, replace=False)
            cp[a], cp[b] = cp[b], cp[a]
        elif op == "ins":
            a, b = rng.choice(N, 2, replace=False)
            g = cp[a]
            cp = np.delete(cp, a)
            cp = np.insert(cp, b, g)
        elif op == "blk":
            L = int(rng.integers(2, 4))
            a = int(rng.integers(0, N - L))
            blk = cp[a:a+L].copy()
            rest = np.concatenate([cp[:a], cp[a+L:]])
            b = int(rng.integers(0, len(rest) + 1))
            cp = np.concatenate([rest[:b], blk, rest[b:]])
        elif op == "pri":
            i, j = sorted(rng.choice(N, 2, replace=False))
            if WGT_PRI[cp[j]] <= WGT_PRI[cp[i]]:
                i, j = j, i
            cp[i], cp[j] = cp[j], cp[i]
        else:
            k = rng.integers(0, N)
            cc[k] = ~cc[k]

        cf = budget.evaluate(cp, cc)
        d = cf[0] - cur_f[0]
        acc = d < 0 or rng.random() < np.exp(-d / max(T, 1e-12))

        wts[oi] = lam*wts[oi] + (1-lam)*(1.0 if d < 0 else (0.3 if acc else 0.0))

        if acc:
            cur_p, cur_c, cur_f = cp, cc, cf
            if cf[0] < best[2][0]:
                best = (cp.copy(), cc.copy(), cf)
                stag = 0
            else:
                stag += 1
        else:
            stag += 1

        if reheat and stag >= stag_limit:
            T = min(T0, T * rf)
            cur_p, cur_c, cur_f = best[0].copy(), best[1].copy(), best[2]
            stag = 0

        T *= alpha

    return float(best[2][1][0]), budget.evals


def _quench_only(seed, max_evals=MAX_EVALS, T0=0.02, alpha=0.999,
                 quench_frac=QUENCH_FRAC):
    rng = np.random.default_rng(seed)
    budget = CGKBudget(max_evals)

    cur_p = rng.permutation(N)
    cur_c = rng.random(N) < P_CANCEL_INIT
    cur_f = budget.evaluate(cur_p, cur_c)
    best = (cur_p.copy(), cur_c.copy(), cur_f)

    T = T0
    quenched = False

    while budget.remaining > 0:
        if not quenched and budget.evals >= quench_frac * max_evals:
            T = 1e-12
            quenched = True
            cur_p, cur_c, cur_f = best[0].copy(), best[1].copy(), best[2]

        cp, cc = cur_p.copy(), cur_c.copy()
        u = rng.random()

        if u < 0.45:
            a, b = rng.choice(N, 2, replace=False)
            cp[a], cp[b] = cp[b], cp[a]
        elif u < 0.90:
            a, b = rng.choice(N, 2, replace=False)
            g = cp[a]
            cp = np.delete(cp, a)
            cp = np.insert(cp, b, g)
        else:
            k = rng.integers(0, N)
            cc[k] = ~cc[k]

        cf = budget.evaluate(cp, cc)
        d = cf[0] - cur_f[0]

        if d < 0 or rng.random() < np.exp(-d / max(T, 1e-12)):
            cur_p, cur_c, cur_f = cp, cc, cf
            best = _update_best(best, cp, cc, cf)

        T *= alpha

    return float(best[2][1][0]), budget.evals


# -------------------------- BENCHMARK FUNCTIONS --------------------------
def sphere(x):
    return float(np.sum(x*x))

def rosenbrock(x):
    return float(np.sum(100*(x[1:] - x[:-1]**2)**2 + (1-x[:-1])**2))

def rastrigin(x):
    return float(10*len(x) + np.sum(x*x - 10*np.cos(2*np.pi*x)))

def ackley(x):
    n = len(x)
    return float(-20*np.exp(-0.2*np.sqrt(np.sum(x*x)/n))
                 - np.exp(np.sum(np.cos(2*np.pi*x))/n) + 20 + np.e)

def griewank(x):
    return float(np.sum(x*x)/4000 - np.prod(np.cos(x/np.sqrt(np.arange(1, len(x)+1)))) + 1)

def schwefel(x):
    return float(418.9829*len(x) - np.sum(x*np.sin(np.sqrt(np.abs(x)))))

def levy(x):
    w = 1 + (x-1)/4
    return float(np.sin(np.pi*w[0])**2
                 + np.sum((w[:-1]-1)**2*(1+10*np.sin(np.pi*w[:-1]+1)**2))
                 + (w[-1]-1)**2*(1+np.sin(2*np.pi*w[-1])**2))

def zakharov(x):
    s = np.sum(0.5*np.arange(1, len(x)+1)*x)
    return float(np.sum(x*x) + s**2 + s**4)


FUNCS = {
    "F1 Sphere": (sphere, -100, 100),
    "F2 Rosenbrock": (rosenbrock, -30, 30),
    "F3 Rastrigin": (rastrigin, -5.12, 5.12),
    "F4 Ackley": (ackley, -32, 32),
    "F5 Griewank": (griewank, -600, 600),
    "F6 Schwefel": (schwefel, -500, 500),
    "F7 Levy": (levy, -10, 10),
    "F8 Zakharov": (zakharov, -10, 10),
}


class ObjBudget:
    def __init__(self, f, max_evals=BENCH_MAX_EVALS, grid=BENCH_GRID):
        self.f = f
        self.max_evals = int(max_evals)
        self.evals = 0
        self.best = np.inf
        self.grid = np.asarray(grid, int)
        self.points = []
        self.gi = 0

    @property
    def remaining(self):
        return self.max_evals - self.evals

    def evaluate(self, x):
        if self.evals >= self.max_evals:
            raise RuntimeError("Benchmark evaluation budget exhausted")
        y = float(self.f(x))
        self.evals += 1
        if y < self.best:
            self.best = y
        while self.gi < len(self.grid) and self.evals >= self.grid[self.gi]:
            self.points.append(self.best)
            self.gi += 1
        return y

    def curve(self):
        while len(self.points) < len(self.grid):
            self.points.append(self.best)
        return np.asarray(self.points, float)


def _t0_estimate_counted(budget, x, lo, hi, rng, n=30):
    vals = []
    for _ in range(min(n, budget.remaining)):
        xn = np.clip(x + rng.normal(0, 0.1*(hi-lo), BD), lo, hi)
        vals.append(budget.evaluate(xn))
    return max(1e-12, float(np.std(vals))) if vals else 1e-12


def bench_sa(f, lo, hi, seed, max_evals=BENCH_MAX_EVALS):
    rng = np.random.default_rng(seed)
    b = ObjBudget(f, max_evals)

    x = rng.uniform(lo, hi, BD)
    fx = b.evaluate(x)
    T = _t0_estimate_counted(b, x, lo, hi, rng)
    alpha = (1e-8)**(1.0/max(max_evals, 1))

    while b.remaining > 0:
        xn = np.clip(x + rng.normal(0, 0.1*(hi-lo), BD), lo, hi)
        fn = b.evaluate(xn)
        d = fn-fx
        if d < 0 or rng.random() < np.exp(-d/max(T, 1e-300)):
            x, fx = xn, fn
        T *= alpha

    return b.best, b.curve(), b.evals


def bench_amqsa(f, lo, hi, seed, max_evals=BENCH_MAX_EVALS):
    rng = np.random.default_rng(seed)
    b = ObjBudget(f, max_evals)

    R = hi-lo
    x = rng.uniform(lo, hi, BD)
    fx = b.evaluate(x)
    bx, bf = x.copy(), fx
    T = _t0_estimate_counted(b, x, lo, hi, rng)
    alpha = (1e-8)**(1.0/max(max_evals, 1))
    wts = np.ones(4)
    quenched = False

    while b.remaining > 0:
        if not quenched and b.evals >= QUENCH_FRAC*max_evals:
            T = 0.0
            quenched = True
            x, fx = bx.copy(), bf

        pr = np.maximum(wts, 0.05)
        pr = pr/pr.sum()
        oi = rng.choice(4, p=pr)

        if oi == 0:
            xn = x + rng.normal(0, 0.10*R, BD)
        elif oi == 1:
            xn = x + rng.normal(0, 0.01*R, BD)
        elif oi == 2:
            xn = x.copy()
            k = rng.integers(0, BD)
            xn[k] += rng.normal(0, 0.10*R)
        else:
            xn = x + 0.05*R*rng.standard_cauchy(BD)

        xn = np.clip(xn, lo, hi)
        fn = b.evaluate(xn)
        d = fn-fx
        acc = d < 0 or (T > 0 and rng.random() < np.exp(-d/max(T, 1e-300)))
        wts[oi] = 0.9*wts[oi] + 0.1*(1.0 if d < 0 else (0.3 if acc else 0.0))

        if acc:
            x, fx = xn, fn
            if fn < bf:
                bx, bf = xn.copy(), fn

        T *= alpha

    return b.best, b.curve(), b.evals


def bench_ga(f, lo, hi, seed, max_evals=BENCH_MAX_EVALS):
    rng = np.random.default_rng(seed)
    b = ObjBudget(f, max_evals)

    pop = rng.uniform(lo, hi, (BPOP, BD))
    fits = np.array([b.evaluate(p) for p in pop])

    while b.remaining >= BPOP:
        new = []
        for _ in range(BPOP):
            i1 = min(rng.choice(BPOP, 3, replace=False), key=lambda i: fits[i])
            i2 = min(rng.choice(BPOP, 3, replace=False), key=lambda i: fits[i])
            beta = rng.uniform(-0.25, 1.25, BD)
            child = beta*pop[i1] + (1-beta)*pop[i2]
            mmask = rng.random(BD) < 0.1
            child = child + mmask*rng.normal(0, 0.05*(hi-lo), BD)
            new.append(np.clip(child, lo, hi))

        pop = np.array(new)
        fits = np.array([b.evaluate(p) for p in pop])

    # BENCH_MAX_EVALS=10,000 and BPOP=30 leaves 10 evaluations after full batches.
    # Use the remainder on independently generated offspring so the total is exact.
    while b.remaining > 0:
        i1 = min(rng.choice(BPOP, 3, replace=False), key=lambda i: fits[i])
        i2 = min(rng.choice(BPOP, 3, replace=False), key=lambda i: fits[i])
        beta = rng.uniform(-0.25, 1.25, BD)
        child = beta*pop[i1] + (1-beta)*pop[i2]
        child = np.clip(child, lo, hi)
        b.evaluate(child)

    return b.best, b.curve(), b.evals


def bench_pso(f, lo, hi, seed, max_evals=BENCH_MAX_EVALS):
    rng = np.random.default_rng(seed)
    b = ObjBudget(f, max_evals)

    x = rng.uniform(lo, hi, (BPOP, BD))
    v = np.zeros_like(x)
    fits = np.array([b.evaluate(p) for p in x])

    pb, pf = x.copy(), fits.copy()
    gb = x[fits.argmin()].copy()

    it = 0
    approx_iters = max(1, math.ceil((max_evals-BPOP)/BPOP))

    while b.remaining > 0:
        w = 0.9 - 0.5*min(it/approx_iters, 1.0)
        r1, r2 = rng.random((BPOP, BD)), rng.random((BPOP, BD))
        v = np.clip(
            w*v + 1.8*r1*(pb-x) + 1.8*r2*(gb-x),
            -0.2*(hi-lo), 0.2*(hi-lo)
        )
        x = np.clip(x+v, lo, hi)

        m = min(BPOP, b.remaining)
        for i in range(m):
            fi = b.evaluate(x[i])
            if fi < pf[i]:
                pb[i], pf[i] = x[i].copy(), fi

        gb = pb[pf.argmin()].copy()
        it += 1

    return b.best, b.curve(), b.evals


def bench_zoa(f, lo, hi, seed, max_evals=BENCH_MAX_EVALS):
    rng = np.random.default_rng(seed)
    b = ObjBudget(f, max_evals)

    x = rng.uniform(lo, hi, (BPOP, BD))
    fits = np.array([b.evaluate(p) for p in x])
    pz = x[fits.argmin()].copy()
    pzf = fits.min()

    while b.remaining > 0:
        for i in range(BPOP):
            if b.remaining <= 0:
                break

            r = rng.random(BD)
            I = 1 + rng.integers(0, 2)
            xn = np.clip(x[i] + r*(pz-I*x[i]), lo, hi)
            fn = b.evaluate(xn)
            if fn < fits[i]:
                x[i], fits[i] = xn, fn
            if fits[i] < pzf:
                pz, pzf = x[i].copy(), fits[i]

            if b.remaining <= 0:
                break

            progress = b.evals/max_evals
            if rng.random() < 0.5:
                xn = np.clip(
                    x[i] + 0.01*(2*rng.random(BD)-1)*(1-progress)*x[i],
                    lo, hi
                )
            else:
                k = rng.integers(0, BPOP)
                r = rng.random(BD)
                I = 1 + rng.integers(0, 2)
                xn = np.clip(x[i] + r*(x[k]-I*x[i]), lo, hi)

            fn = b.evaluate(xn)
            if fn < fits[i]:
                x[i], fits[i] = xn, fn
            if fits[i] < pzf:
                pz, pzf = x[i].copy(), fits[i]

    return b.best, b.curve(), b.evals


BENCH_ALGOS = {
    "SA": bench_sa,
    "GA": bench_ga,
    "PSO": bench_pso,
    "ZOA": bench_zoa,
    "AMQSA": bench_amqsa,
}


# -------------------------- STATISTICS UTILITIES --------------------------
def bootstrap_mean_ci(x, level=0.95, B=5000, seed=12345):
    x = np.asarray(x, float)
    rng = np.random.default_rng(seed)
    means = np.empty(B)
    n = len(x)
    for i in range(B):
        means[i] = rng.choice(x, size=n, replace=True).mean()
    a = (1-level)/2
    return np.quantile(means, [a, 1-a])


def holm_adjust(pvals):
    pvals = np.asarray(pvals, float)
    m = len(pvals)
    order = np.argsort(pvals)
    adj_sorted = np.empty(m)
    running = 0.0
    for k, idx in enumerate(order):
        val = (m-k)*pvals[idx]
        running = max(running, val)
        adj_sorted[k] = min(1.0, running)
    out = np.empty(m)
    for k, idx in enumerate(order):
        out[idx] = adj_sorted[k]
    return out


def paired_rank_biserial(reference, comparator):
    # Positive = reference is better for minimization.
    d = np.asarray(comparator, float) - np.asarray(reference, float)
    d = d[d != 0]
    if len(d) == 0:
        return 0.0
    r = rankdata(np.abs(d))
    wp = r[d > 0].sum()
    wm = r[d < 0].sum()
    return float((wp-wm)/(wp+wm))


print("Definitions loaded.")
print(f"Instance: {N} flights | {DATA_SOURCE}")
print(f"CGK fair budget: {MAX_EVALS:,} total objective evaluations / run")
print(f"Benchmark fair budget: {BENCH_MAX_EVALS:,} total objective evaluations / run")


In [ ]:

# STEP L4 — Exact dynamic programming + heuristic optimality gaps

import numpy as np, pandas as pd, time, json, zipfile

FULL_FLIGHTS = FLIGHTS.sort_values("sched_min_from_1800", kind="stable").reset_index(drop=True).copy()
FULL_DATA_SOURCE = DATA_SOURCE

SIZES = [8, 10, 12, 14, 16]
SEEDS = list(range(2000,2020))
ALGOS = ["SA", "VNS", "AMQSA"]

# ------------------------------------------------------------
# Standard permutation-native VNS from Phase 3B
# ------------------------------------------------------------
def shake_neighbor(rng, perm, cancel, k):
    p=perm.copy()
    c=cancel.copy()

    if k==0:
        a,b=rng.choice(N,2,replace=False)
        p[a],p[b]=p[b],p[a]
    elif k==1:
        a,b=rng.choice(N,2,replace=False)
        g=p[a]
        p=np.delete(p,a)
        p=np.insert(p,b,g)
    elif k==2:
        L=int(rng.integers(2,4))
        a=int(rng.integers(0,N-L))
        blk=p[a:a+L].copy()
        rest=np.concatenate([p[:a],p[a+L:]])
        b=int(rng.integers(0,len(rest)+1))
        p=np.concatenate([rest[:b],blk,rest[b:]])
    else:
        j=rng.integers(0,N)
        c[j]=~c[j]

    return p,c


def run_vns(seed, max_evals=MAX_EVALS):
    rng=np.random.default_rng(seed)
    budget=CGKBudget(max_evals)

    cur_p=rng.permutation(N)
    cur_c=rng.random(N)<P_CANCEL_INIT
    cur_f=budget.evaluate(cur_p,cur_c)
    best=(cur_p.copy(),cur_c.copy(),cur_f)
    budget.observe(best)

    k=0
    while budget.remaining>0:
        cp,cc=shake_neighbor(rng,cur_p,cur_c,k)
        cf=budget.evaluate(cp,cc)

        if cf[0] < cur_f[0]:
            cur_p,cur_c,cur_f=cp,cc,cf
            best=_update_best(best,cp,cc,cf)
            k=0
        else:
            k=(k+1)%4

        budget.observe(best)

    assert budget.evals==max_evals
    return best,budget.curve(best),budget.evals


# ------------------------------------------------------------
# Activate a deterministic small subset
# ------------------------------------------------------------
def activate_subset(n):
    global FLIGHTS,N,SCHED,CAP,NP_SEATS,CFM,FARE_V,REPOS,WAKE,ENG,SID,SEP_M,F_REF,W,WGT_PRI

    FLIGHTS = FULL_FLIGHTS.iloc[:n].copy().reset_index(drop=True)
    N = len(FLIGHTS)

    SCHED = FLIGHTS["sched_min_from_1800"].to_numpy(float)
    CAP = FLIGHTS["capacity"].to_numpy(float)
    NP_SEATS = 0.75*CAP
    CFM = FLIGHTS["cfm_idr_per_min"].to_numpy(float)
    FARE_V = FLIGHTS["fare_idr"].to_numpy(float)
    REPOS = FLIGHTS["reposition_idr"].to_numpy(float)
    WAKE = FLIGHTS["wake"].to_numpy()
    ENG = FLIGHTS["engine"].to_numpy()
    SID = FLIGHTS["sid"].to_numpy()

    SEP_M = np.array([[_sep_rule(u,v) for v in range(N)] for u in range(N)])

    # Same normalization protocol as the authoritative study.
    rr=np.random.default_rng(0)
    samples=[]
    for _ in range(300):
        p=rr.permutation(N)
        c=rr.random(N)<0.05
        samples.append(decode(p,c)[:3])

    F_REF=np.asarray(samples,float).mean(axis=0)
    W=np.array([1/3,1/3,1/3],float)
    WGT_PRI=NP_SEATS*CP + CFM + NP_SEATS*FARE_V/300.0

    FLIGHTS.to_csv(OUTPUT_DIR/f"phase3f_instance_N{N}.csv",index=False)


# ------------------------------------------------------------
# Exact dynamic programming
# ------------------------------------------------------------
def exact_scalar_dp():
    """
    Exact global optimum for the current small instance.

    State:
      (operated_mask, last_operated_flight, current_departure_half_min_tick)

    Objective:
      Start from the all-cancel scalar objective.
      Operating a flight replaces its fixed cancellation contribution with its
      operated contribution at the exact decoder departure time.

    Any flight not included in the operated path is cancelled.

    Dominance:
      For the same (mask,last), a state is discarded if another state has both
      earlier/equal time and lower/equal accumulated scalar cost.
    """
    unit=0.5
    reopen_tick=int(round(REOPEN/unit))
    cutoff_tick=int(round(RECOVERY_CUTOFF/unit))
    sep_tick=np.rint(SEP_M/unit).astype(int)

    # Fixed scalar contribution if flight j is cancelled.
    can_f1=(COMP_CANCEL + REV_LOSS_FRAC*FARE_V)*NP_SEATS + REPOS
    can_f2=PAX_CANCEL_LOSS*NP_SEATS
    can_f3=np.full(N,CANCEL_EQUIV_MIN,float)

    can_scalar=(
        W[0]*can_f1/F_REF[0] +
        W[1]*can_f2/F_REF[1] +
        W[2]*can_f3/F_REF[2]
    )

    all_cancel_scalar=float(can_scalar.sum())

    # Cache the operated-minus-cancel scalar delta for each (flight,tick).
    delta_cache={}

    def operated_delta(j,tick):
        key=(int(j),int(tick))
        if key in delta_cache:
            return delta_cache[key]

        dep=tick*unit
        delay=dep-SCHED[j]

        f1=delay*CFM[j] + mu_disappointment(delay)*NP_SEATS[j]*FARE_V[j]
        f2=NP_SEATS[j]*CP*delay
        f3=delay

        op_scalar=(
            W[0]*f1/F_REF[0] +
            W[1]*f2/F_REF[1] +
            W[2]*f3/F_REF[2]
        )

        val=float(op_scalar-can_scalar[j])
        delta_cache[key]=val
        return val

    # Initial one-flight operated states.
    layer={}
    parent={}

    for j in range(N):
        key=(1<<j,j,reopen_tick)
        layer[key]=all_cancel_scalar+operated_delta(j,reopen_tick)
        parent[key]=None

    best_cost=all_cancel_scalar
    best_key=None
    max_layer_states=len(layer)
    total_retained_states=len(layer)

    if layer:
        kmin=min(layer,key=layer.get)
        if layer[kmin] < best_cost:
            best_cost=layer[kmin]
            best_key=kmin

    # Add one operated flight at each layer.
    for operated_count in range(1,N):
        candidates={}
        candidate_parent={}

        for key,cost in layer.items():
            mask,last,tick=key
            remaining=((1<<N)-1)^mask

            while remaining:
                lsb=remaining & -remaining
                j=lsb.bit_length()-1
                remaining-=lsb

                new_tick=tick+sep_tick[last,j]
                if new_tick > cutoff_tick:
                    continue

                new_mask=mask|lsb
                new_key=(new_mask,j,new_tick)
                new_cost=cost+operated_delta(j,new_tick)

                if new_cost < candidates.get(new_key,float("inf")):
                    candidates[new_key]=new_cost
                    candidate_parent[new_key]=key

        # Exact dominance pruning within each (mask,last).
        grouped={}
        for key,cost in candidates.items():
            mask,last,tick=key
            grouped.setdefault((mask,last),[]).append((tick,cost,key))

        next_layer={}
        for _,arr in grouped.items():
            arr.sort(key=lambda x:x[0])  # increasing departure time
            best_cost_so_far=float("inf")

            for tick,cost,key in arr:
                # Retain only nondominated time-cost frontier states.
                if cost < best_cost_so_far - 1e-15:
                    next_layer[key]=cost
                    parent[key]=candidate_parent[key]
                    best_cost_so_far=cost

        layer=next_layer
        max_layer_states=max(max_layer_states,len(layer))
        total_retained_states+=len(layer)

        if not layer:
            break

        kmin=min(layer,key=layer.get)
        if layer[kmin] < best_cost:
            best_cost=layer[kmin]
            best_key=kmin

    # Reconstruct exact operated path.
    operated=[]
    key=best_key
    while key is not None:
        operated.append(key[1])
        key=parent[key]
    operated=operated[::-1]

    cancel=np.ones(N,dtype=bool)
    if operated:
        cancel[np.asarray(operated,int)]=False

    # A complete permutation for the submitted decoder:
    # operated flights first in optimal order, cancelled complement afterwards.
    cancelled=[j for j in range(N) if cancel[j]]
    perm=np.asarray(operated+cancelled,int)

    decoded=fitness(perm,cancel)

    # Exact DP and submitted decoder must agree.
    if not np.isclose(best_cost,decoded[0],rtol=0,atol=1e-10):
        raise RuntimeError(
            f"Exact-DP/decoder mismatch: DP={best_cost}, decoder={decoded[0]}"
        )

    return {
        "scalar_optimum":float(best_cost),
        "f1_IDR":float(decoded[1][0]),
        "f2_IDR":float(decoded[1][1]),
        "f3_min":float(decoded[1][2]),
        "cancelled":int(decoded[2]),
        "operated_order":operated,
        "cancel_mask":cancel,
        "perm":perm,
        "max_layer_states":int(max_layer_states),
        "total_retained_states":int(total_retained_states),
    }


def run_one(algo,seed):
    if algo=="SA":
        return run_sa(seed,max_evals=MAX_EVALS)
    if algo=="VNS":
        return run_vns(seed,max_evals=MAX_EVALS)
    if algo=="AMQSA":
        return run_amqsa(seed,max_evals=MAX_EVALS)
    raise ValueError(algo)


exact_rows=[]
heur_rows=[]
sequence_rows=[]
t_all=time.perf_counter()

for n in SIZES:
    print(f"\n================ N={n} ================")
    activate_subset(n)

    # ---- exact solve ----
    t0=time.perf_counter()
    ex=exact_scalar_dp()
    exact_runtime=time.perf_counter()-t0

    exact_rows.append({
        "N":n,
        "SubsetRule":"earliest N scheduled flights from authoritative seed-7 bank",
        "ExactScalarOptimum":ex["scalar_optimum"],
        "Exact_f1_IDR":ex["f1_IDR"],
        "Exact_f2_IDR":ex["f2_IDR"],
        "Exact_f3_min":ex["f3_min"],
        "ExactCancelled":ex["cancelled"],
        "ExactRuntime_s":exact_runtime,
        "MaxLayerStates":ex["max_layer_states"],
        "TotalRetainedStates":ex["total_retained_states"],
    })

    for pos,fl in enumerate(ex["operated_order"],start=1):
        sequence_rows.append({
            "N":n,
            "OperatedPosition":pos,
            "FlightIndex":int(fl),
            "Flight":FLIGHTS.iloc[fl]["flight"],
            "Scheduled_min_from_1800":float(SCHED[fl]),
            "SID":SID[fl],
            "Wake":WAKE[fl],
            "Engine":ENG[fl],
        })

    print(
        f"Exact scalar={ex['scalar_optimum']:.9f} | "
        f"f1={ex['f1_IDR']/1e9:.6f} B | "
        f"cancel={ex['cancelled']} | runtime={exact_runtime:.2f}s"
    )

    # ---- heuristic runs ----
    for algo in ALGOS:
        local=[]

        for run_i,seed in enumerate(SEEDS,start=1):
            h0=time.perf_counter()
            best,curve,nevals=run_one(algo,seed)
            rt=time.perf_counter()-h0

            p,c,(scalar,fvec,ncan)=best
            gap=(scalar-ex["scalar_optimum"])/ex["scalar_optimum"]*100.0

            heur_rows.append({
                "N":n,
                "Algorithm":algo,
                "Run":run_i,
                "Seed":seed,
                "ScalarFitness":scalar,
                "OptimalityGapPct":gap,
                "f1_IDR":fvec[0],
                "f2_IDR":fvec[1],
                "f3_min":fvec[2],
                "Cancelled":ncan,
                "Evaluations":nevals,
                "Runtime_s":rt,
            })
            local.append((scalar,gap))

        a=np.asarray(local,float)
        print(
            f"{algo:5s} | scalar mean={a[:,0].mean():.9f} "
            f"| mean gap={a[:,1].mean():.5f}% "
            f"| best gap={a[:,1].min():.5f}%"
        )

exact_df=pd.DataFrame(exact_rows)
heur_raw=pd.DataFrame(heur_rows)
seq_df=pd.DataFrame(sequence_rows)

exact_df.to_csv(OUTPUT_DIR/"phase3f_exact_results.csv",index=False)
heur_raw.to_csv(OUTPUT_DIR/"phase3f_heuristic_raw.csv",index=False)
seq_df.to_csv(OUTPUT_DIR/"phase3f_exact_sequences.csv",index=False)

assert heur_raw.Evaluations.eq(MAX_EVALS).all(),"Evaluation-budget violation"
assert (heur_raw.OptimalityGapPct >= -1e-8).all(),"A heuristic appears better than the exact optimum."

# ------------------------------------------------------------
# Gap summary
# ------------------------------------------------------------
summary=(
    heur_raw.groupby(["N","Algorithm"],as_index=False)
    .agg(
        Scalar_mean=("ScalarFitness","mean"),
        Scalar_std=("ScalarFitness","std"),
        Gap_mean_pct=("OptimalityGapPct","mean"),
        Gap_median_pct=("OptimalityGapPct","median"),
        Gap_best_pct=("OptimalityGapPct","min"),
        Gap_worst_pct=("OptimalityGapPct","max"),
        Runtime_mean_s=("Runtime_s","mean"),
        f1_mean_IDR=("f1_IDR","mean"),
    )
)

# Count exact hits with a strict but numerical-safe tolerance.
hit_rows=[]
for n in SIZES:
    opt=exact_df.loc[exact_df.N==n,"ExactScalarOptimum"].iloc[0]
    for algo in ALGOS:
        g=heur_raw[(heur_raw.N==n)&(heur_raw.Algorithm==algo)]
        hits=int(np.isclose(g.ScalarFitness,opt,rtol=0,atol=1e-9).sum())
        hit_rows.append({"N":n,"Algorithm":algo,"ExactHits":hits,"Runs":len(g)})

hits=pd.DataFrame(hit_rows)
summary=summary.merge(hits,on=["N","Algorithm"],how="left")
summary.to_csv(OUTPUT_DIR/"phase3f_optimality_gap_summary.csv",index=False)

print("\n=== EXACT RESULTS ===")
display(exact_df.round(6))

print("\n=== HEURISTIC OPTIMALITY GAPS ===")
display(summary.round(6))

# Plot mean optimality gap
fig,ax=plt.subplots(figsize=(8.5,5))
for algo in ALGOS:
    d=summary[summary.Algorithm==algo]
    ax.plot(d.N,d.Gap_mean_pct,marker="o",label=algo)
ax.set_xlabel("Small-instance size N")
ax.set_ylabel("Mean scalar optimality gap (%)")
ax.set_title("Exact small-instance optimality gaps")
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
fig.savefig(OUTPUT_DIR/"phase3f_optimality_gap.png",dpi=220,bbox_inches="tight")
plt.show()

# Plot exact-solver runtime
fig,ax=plt.subplots(figsize=(8.5,5))
ax.plot(exact_df.N,exact_df.ExactRuntime_s,marker="o")
ax.set_xlabel("Small-instance size N")
ax.set_ylabel("Exact DP runtime (s)")
ax.set_title("Exact dynamic-programming runtime")
ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(OUTPUT_DIR/"phase3f_exact_runtime.png",dpi=220,bbox_inches="tight")
plt.show()

completion={
    "completed":True,
    "sizes":SIZES,
    "subset_rule":"earliest N scheduled flights from authoritative seed-7 bank",
    "algorithms":ALGOS,
    "runs_per_algorithm_size":20,
    "evaluations_per_heuristic_run":MAX_EVALS,
    "exact_method":"dynamic programming over operated subset, last flight, and 0.5-min departure tick with exact dominance pruning",
    "primary_endpoint":"normalized scalar fitness",
    "all_decoder_consistency_checks_passed":True,
    "all_budget_checks_passed":True,
    "elapsed_seconds":time.perf_counter()-t_all,
}

with open(OUTPUT_DIR/"phase3f_completion.json","w") as f:
    json.dump(completion,f,indent=2)

zip_path=Path("/content/AMQSA_PHASE3F_EXACT_OPTIMALITY_GAP_RESULTS.zip")
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for p in OUTPUT_DIR.rglob("*"):
        if p.is_file():
            z.write(p,arcname=p.relative_to(OUTPUT_DIR))

print("\nALL HEURISTIC RUNS USED EXACTLY 5,000 EVALUATIONS.")
print("EXACT-DP / SUBMITTED-DECODER CONSISTENCY CHECKS PASSED.")
print(f"PHASE 3F COMPLETE in {completion['elapsed_seconds']/60:.1f} minutes.")
print("ZIP:",zip_path)



## Upload after completion

The run is complete only if the final cell prints:

```text
ALL HEURISTIC RUNS USED EXACTLY 5,000 EVALUATIONS.
EXACT-DP / SUBMITTED-DECODER CONSISTENCY CHECKS PASSED.
PHASE 3F COMPLETE ...
```

Upload:

1. `AMQSA_PHASE3F_EXACT_OPTIMALITY_GAP_RESULTS.zip`
2. this executed notebook.

The exact optimum applies only to the tested small nested subsets. It must not be extrapolated as an optimality certificate for the full N=48 case.
